<a href="https://colab.research.google.com/github/Gabalecrim/BolaBarraControlSystem/blob/main/IA_Aula_5_buscas_informadas_gulosa_a_estrela.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Busca informada: Busca Gulosa e A*

Este notebook extende os conceitos trabalhados na aula anterior de **busca cega** para **busca informada**.

Aqui, usamos a **mesma ideia de modelagem do problema**, mas agora adicionamos uma **heurística** para orientar a busca.
O objetivo é comparar, no mesmo exemplo:

- **Busca Gulosa (Greedy Best-First Search)**
- **Busca A\***

## Ideia central

Na busca informada, além do custo já percorrido, podemos usar uma estimativa de quão perto um estado está do objetivo.

- **Busca Gulosa:** escolhe expandir o nó que parece mais promissor pela heurística `h(n)`.
- **A\*:** escolhe o nó com menor valor de `f(n) = g(n) + h(n)`, onde:
  - `g(n)` = custo acumulado até o nó
  - `h(n)` = estimativa do custo restante até o objetivo

## Objetivo didático

Ao final, você poderá observar que:

- a **busca gulosa** costuma ser mais "apressada" na direção do objetivo;
- a **A\*** tende a ser mais confiável quando a heurística é adequada;
- os dois algoritmos podem chegar a caminhos diferentes.


## Exemplo utilizado

Vamos manter o contexto do mapa da **Península Ibérica**, usado na aula anterior.

Além das **transições** e dos **custos**, agora cada cidade também terá um valor heurístico, que representa uma **estimativa de distância até Barcelona**.

> Quanto menor o valor heurístico, mais "promissor" o estado parece para a busca informada.


In [ ]:

from dataclasses import dataclass
import heapq
from itertools import count
from math import inf


# =========================================================
# ESTRUTURAS BÁSICAS DO PROBLEMA
# =========================================================

@dataclass(frozen=True)
class Acao:
    """
    Representa uma ação possível a partir de um estado.

    Exemplo:
    - "norte"
    - "sul"
    - "leste"
    - "oeste"
    """
    nome: str

    def __str__(self):
        return self.nome


@dataclass(frozen=True)
class Estado:
    """
    Representa um estado do problema de busca.

    Neste notebook, cada estado será uma cidade.
    """
    nome: str

    def __str__(self):
        return self.nome


class Problema:
    """
    Modela um problema de busca informado.

    Atributos principais:
    - estado_inicial: onde a busca começa
    - estados_objetivos: conjunto de estados-meta
    - transicoes: dicionário no formato
        {
            'Cidade A': {'acao 1': Estado(...), 'acao 2': Estado(...)},
            ...
        }
    - custos: dicionário com o custo de cada ação
    - heuristicas: dicionário com o valor heurístico de cada estado

    Observação:
    Se os custos não forem informados, assumimos custo 1 para todas as ações.
    Se as heurísticas não forem informadas, assumimos heurística 0 para todos os estados.
    """

    def __init__(self, estado_inicial, estados_objetivos, transicoes, custos=None, heuristicas=None):
        self.estado_inicial = estado_inicial
        self.estados_objetivos = set(estados_objetivos)
        self.transicoes = transicoes
        self.infinito = float("inf")

        # Se os custos não forem passados, todas as ações custam 1.
        if custos is None:
            self.custos = {
                nome_estado: {nome_acao: 1 for nome_acao in acoes.keys()}
                for nome_estado, acoes in transicoes.items()
            }
        else:
            self.custos = custos

        # Se não houver heurística, todos os estados recebem 0.
        self.heuristicas = heuristicas or {}

    def e_objetivo(self, estado):
        """Retorna True se o estado for um dos objetivos."""
        return estado in self.estados_objetivos

    def acoes_disponiveis(self, estado):
        """Retorna a lista de ações disponíveis em um estado."""
        return list(self.transicoes.get(estado.nome, {}).keys())

    def resultado(self, estado, acao):
        """Aplica uma ação em um estado e devolve o próximo estado."""
        return self.transicoes.get(estado.nome, {}).get(acao.nome)

    def custo_acao(self, estado, acao):
        """Retorna o custo da ação a partir do estado informado."""
        return self.custos.get(estado.nome, {}).get(acao.nome, self.infinito)

    def heuristica(self, estado):
        """
        Retorna o valor heurístico do estado.

        Quanto menor esse valor, mais perto do objetivo acreditamos estar.
        """
        return self.heuristicas.get(estado.nome, 0)


class No:
    """
    Representa um nó da árvore de busca.

    Cada nó guarda:
    - estado atual
    - pai
    - ação usada para chegar nele
    - custo acumulado desde a raiz (g)
    - profundidade
    """

    def __init__(self, estado, pai=None, acao=None, custo=0, profundidade=0):
        self.estado = estado
        self.pai = pai
        self.acao = acao
        self.custo = custo              # g(n)
        self.profundidade = profundidade

    def __str__(self):
        return f"{self.estado.nome} (g={self.custo}, profundidade={self.profundidade})"

    def expandir(self, problema):
        """
        Gera os nós filhos a partir do estado atual.

        Para cada ação disponível:
        1. descobre o próximo estado
        2. calcula o novo custo acumulado
        3. cria o nó filho
        """
        filhos = []

        for nome_acao in problema.acoes_disponiveis(self.estado):
            acao = Acao(nome_acao)
            proximo_estado = problema.resultado(self.estado, acao)

            if proximo_estado is None:
                continue

            custo_total = self.custo + problema.custo_acao(self.estado, acao)

            filho = No(
                estado=proximo_estado,
                pai=self,
                acao=acao,
                custo=custo_total,
                profundidade=self.profundidade + 1
            )
            filhos.append(filho)

        return filhos


# =========================================================
# FUNÇÕES AUXILIARES
# =========================================================

def reconstruir_caminho(no_objetivo):
    """
    Reconstrói o caminho da raiz até o nó objetivo.

    Faz isso voltando pelos ponteiros 'pai' e invertendo a lista no final.
    """
    if no_objetivo is None:
        return []

    caminho = []
    atual = no_objetivo

    while atual is not None:
        caminho.append(atual)
        atual = atual.pai

    caminho.reverse()
    return caminho


def caminho_para_texto(no_objetivo):
    """
    Retorna apenas os nomes dos estados no caminho solução.
    """
    return [no.estado.nome for no in reconstruir_caminho(no_objetivo)]


def mostrar_solucao(no_objetivo, problema=None, nome_algoritmo="Algoritmo"):
    """
    Mostra a solução de forma didática.

    Se o problema for informado, também exibimos h(n) e f(n) em cada estado,
    o que ajuda bastante na aula quando queremos comparar os algoritmos.
    """
    print(f"=== {nome_algoritmo} ===")

    if no_objetivo is None:
        print("Não foi encontrada solução.")
        return

    caminho = reconstruir_caminho(no_objetivo)
    print("Caminho encontrado:")

    for i, no in enumerate(caminho):
        linha = f"{i+1}. Estado: {no.estado.nome} | g(n)={no.custo}"

        if problema is not None:
            h = problema.heuristica(no.estado)
            f = no.custo + h
            linha += f" | h(n)={h} | f(n)={f}"

        print(linha)

        if i < len(caminho) - 1:
            print(f"   Ação aplicada: {caminho[i+1].acao.nome}")

    print()
    print(f"Custo total da solução: {no_objetivo.custo}")
    print(f"Profundidade da solução: {no_objetivo.profundidade}")
    print()


def mostrar_resumo_comparativo(resultados):
    """
    Exibe um resumo comparando os algoritmos.

    'resultados' deve ser um dicionário no formato:
    {
        "Busca Gulosa": {...},
        "Busca A*": {...}
    }
    """
    print("=" * 90)
    print("RESUMO COMPARATIVO")
    print("=" * 90)

    cabecalho = f"{'Algoritmo':<18} {'Caminho':<42} {'Custo':<8} {'Prof.':<8} {'Expandidos':<10}"
    print(cabecalho)
    print("-" * len(cabecalho))

    for nome, dados in resultados.items():
        caminho = " -> ".join(dados["caminho"])
        custo = dados["custo_total"]
        prof = dados["profundidade"]
        expandidos = dados["nos_expandidos"]
        print(f"{nome:<18} {caminho:<42} {custo!s:<8} {prof!s:<8} {expandidos!s:<10}")


# =========================================================
# BUSCA GULOSA
# =========================================================

def busca_gulosa(problema):
    """
    Busca Gulosa (Greedy Best-First Search).

    Ideia:
    - Expande o nó que tiver o menor valor heurístico h(n).
    - Ignora o custo acumulado g(n) no critério de prioridade.

    Consequência:
    - Pode chegar rápido ao objetivo
    - Mas nem sempre encontra o caminho de menor custo
    """
    raiz = No(problema.estado_inicial)

    # heapq organiza a fronteira como fila de prioridade.
    # Cada item será: (prioridade, desempate, no)
    contador = count()
    fronteira = [(problema.heuristica(raiz.estado), next(contador), raiz)]

    explorados = set()
    nos_expandidos = 0

    while fronteira:
        _, _, no_atual = heapq.heappop(fronteira)

        # Se o estado já foi expandido antes, ignoramos.
        if no_atual.estado in explorados:
            continue

        nos_expandidos += 1

        if problema.e_objetivo(no_atual.estado):
            return no_atual, nos_expandidos

        explorados.add(no_atual.estado)

        for filho in no_atual.expandir(problema):
            if filho.estado not in explorados:
                prioridade = problema.heuristica(filho.estado)
                heapq.heappush(fronteira, (prioridade, next(contador), filho))

    return None, nos_expandidos


# =========================================================
# BUSCA A*
# =========================================================

def busca_a_estrela(problema):
    """
    Busca A*.

    Ideia:
    - Expande o nó com menor f(n) = g(n) + h(n).

    Interpretação:
    - g(n): quanto já custou chegar até aqui
    - h(n): quanto ainda estimamos faltar até o objetivo

    Consequência:
    - Costuma ser mais equilibrada do que a busca gulosa
    - Quando h(n) é admissível/consistente, A* encontra solução ótima
    """
    raiz = No(problema.estado_inicial)

    contador = count()
    f_raiz = raiz.custo + problema.heuristica(raiz.estado)
    fronteira = [(f_raiz, next(contador), raiz)]

    # Guarda o menor custo g(n) conhecido para cada estado.
    melhor_custo = {raiz.estado: 0}
    nos_expandidos = 0

    while fronteira:
        _, _, no_atual = heapq.heappop(fronteira)

        # Se apareceu um caminho melhor depois, ignoramos esta versão pior.
        if no_atual.custo > melhor_custo.get(no_atual.estado, inf):
            continue

        nos_expandidos += 1

        if problema.e_objetivo(no_atual.estado):
            return no_atual, nos_expandidos

        for filho in no_atual.expandir(problema):
            novo_custo = filho.custo

            # Só atualizamos se encontramos um caminho melhor para o estado.
            if novo_custo < melhor_custo.get(filho.estado, inf):
                melhor_custo[filho.estado] = novo_custo
                prioridade = novo_custo + problema.heuristica(filho.estado)
                heapq.heappush(fronteira, (prioridade, next(contador), filho))

    return None, nos_expandidos


# =========================================================
# FUNÇÃO GERAL PARA ESCOLHER A ESTRATÉGIA
# =========================================================

def buscar(problema, estrategia="gulosa"):
    """
    Escolhe a estratégia de busca desejada.

    Opções:
    - 'gulosa'
    - 'a*'
    - 'a_estrela'
    - 'astar'
    """
    estrategia = estrategia.lower().strip()

    if estrategia in ("gulosa", "greedy", "busca gulosa"):
        return busca_gulosa(problema)
    elif estrategia in ("a*", "a_estrela", "a estrela", "astar"):
        return busca_a_estrela(problema)
    else:
        raise ValueError("Estratégia inválida. Use 'gulosa' ou 'a*'.")




#Edição de Dados do Problema

In [ ]:
# =========================================================
# EXEMPLO DE USO
# =========================================================

# ---------
# Estados
# ---------
coruna = Estado("A Coruña")
bilbao = Estado("Bilbao")
barcelona = Estado("Barcelona")
lisboa = Estado("Lisboa")
madrid = Estado("Madrid")
valencia = Estado("Valencia")
faro = Estado("Faro")
sevilla = Estado("Sevilla")
granada = Estado("Granada")

# -----------------------------
# Transições entre as cidades
# -----------------------------
viagens = {
    "A Coruña": {"sul": lisboa, "leste": bilbao},
    "Bilbao": {"sul": madrid, "leste": barcelona, "oeste": coruna},
    "Barcelona": {"sul": valencia, "oeste": bilbao},
    "Lisboa": {"norte": coruna, "sul": faro, "leste": madrid},
    "Madrid": {"norte": bilbao, "sul": sevilla, "leste": valencia, "oeste": lisboa},
    "Valencia": {"norte": barcelona, "sul": granada, "oeste": madrid},
    "Faro": {"norte": lisboa, "leste": sevilla},
    "Sevilla": {"norte": madrid, "leste": granada, "oeste": faro},
    "Granada": {"norte": valencia, "oeste": sevilla},
}

# --------------------------
# Custos de cada movimento
# --------------------------
custos_viagem = {
    "A Coruña": {"sul": 4, "leste": 3},
    "Bilbao": {"sul": 2, "leste": 4, "oeste": 3},
    "Barcelona": {"sul": 2, "oeste": 4},
    "Lisboa": {"norte": 4, "sul": 2, "leste": 2},
    "Madrid": {"norte": 2, "sul": 2, "leste": 2, "oeste": 2},
    "Valencia": {"norte": 2, "sul": 3, "oeste": 2},
    "Faro": {"norte": 2, "leste": 1},
    "Sevilla": {"norte": 2, "leste": 4, "oeste": 1},
    "Granada": {"norte": 3, "oeste": 4},
}

# ---------------------------------------------------------
# Heurística: estimativa de distância restante até Barcelona (Objetivo)
# ---------------------------------------------------------
# Valores menores = aparentemente mais perto do objetivo.
# Repare que Granada parece muito promissora pela heurística,
# mas o caminho por ela ficou mais caro.
# Isso foi feito de propósito para destacar a diferença entre
# busca gulosa e A* na comparação didática.
#-----------------------------------------------------------------------------
# Explicação da heurística escolhida:(PREENCHER)

#---------------------------------------------------------------------------
heuristica_barcelona = {
    "A Coruña": 6,
    "Bilbao": 2,
    "Barcelona": 0,
    "Lisboa": 5,
    "Madrid": 2,
    "Valencia": 1,
    "Faro": 5,
    "Sevilla": 3,
    "Granada": 1,
}

#Execução

In [ ]:
objetivos = [barcelona] #Definir aqui qual o destino final

#Declaração dos dados para a função Problema
problema = Problema(
    estado_inicial=faro,
    estados_objetivos=objetivos,
    transicoes=viagens,
    custos=custos_viagem,
    heuristicas=heuristica_barcelona
)

# ---------------------------
# Executando os dois métodos
# ---------------------------
solucao_gulosa, exp_gulosa = buscar(problema, estrategia="gulosa")
solucao_a_estrela, exp_a_estrela = buscar(problema, estrategia="a*")

# ---------------------------
# Mostrando as soluções
# ---------------------------
mostrar_solucao(solucao_gulosa, problema, "Busca Gulosa")
mostrar_solucao(solucao_a_estrela, problema, "Busca A*")

# ---------------------------
# Comparação final
# ---------------------------
resultados = {
    "Busca Gulosa": {
        "caminho": caminho_para_texto(solucao_gulosa),
        "custo_total": solucao_gulosa.custo if solucao_gulosa else None,
        "profundidade": solucao_gulosa.profundidade if solucao_gulosa else None,
        "nos_expandidos": exp_gulosa,
    },
    "Busca A*": {
        "caminho": caminho_para_texto(solucao_a_estrela),
        "custo_total": solucao_a_estrela.custo if solucao_a_estrela else None,
        "profundidade": solucao_a_estrela.profundidade if solucao_a_estrela else None,
        "nos_expandidos": exp_a_estrela,
    }
}

mostrar_resumo_comparativo(resultados)

=== Busca Gulosa ===
Caminho encontrado:
1. Estado: Faro | g(n)=0 | h(n)=5 | f(n)=5
   Ação aplicada: leste
2. Estado: Sevilla | g(n)=1 | h(n)=3 | f(n)=4
   Ação aplicada: leste
3. Estado: Granada | g(n)=5 | h(n)=1 | f(n)=6
   Ação aplicada: norte
4. Estado: Valencia | g(n)=8 | h(n)=1 | f(n)=9
   Ação aplicada: norte
5. Estado: Barcelona | g(n)=10 | h(n)=0 | f(n)=10

Custo total da solução: 10
Profundidade da solução: 4

=== Busca A* ===
Caminho encontrado:
1. Estado: Faro | g(n)=0 | h(n)=5 | f(n)=5
   Ação aplicada: leste
2. Estado: Sevilla | g(n)=1 | h(n)=3 | f(n)=4
   Ação aplicada: norte
3. Estado: Madrid | g(n)=3 | h(n)=2 | f(n)=5
   Ação aplicada: leste
4. Estado: Valencia | g(n)=5 | h(n)=1 | f(n)=6
   Ação aplicada: norte
5. Estado: Barcelona | g(n)=7 | h(n)=0 | f(n)=7

Custo total da solução: 7
Profundidade da solução: 4

RESUMO COMPARATIVO
Algoritmo          Caminho                                    Custo    Prof.    Expandidos
------------------------------------------------

## Leitura da comparação

Neste exemplo, a heurística foi montada para que **Granada pareça muito próxima do objetivo**.  
Com isso:

1. **A busca gulosa tende a se deixar levar pelo menor `h(n)`**  
   Ela escolhe o estado que parece mais promissor no momento, mesmo que o caminho total fique mais caro.

2. **A A\* combina `g(n) + h(n)`**  
   Por isso, ela consegue perceber melhor quando um caminho aparentemente bom pela heurística já acumulou custo demais.

3. **Os caminhos encontrados ficam diferentes**  
   Isso é ótimo para mostrar em aula que:
   - heurística sozinha não garante melhor solução;
   - considerar o custo acumulado faz diferença.

4. **O número de expansões também pode mudar**  
   Às vezes a busca gulosa expande menos nós, mas paga por isso com uma solução pior.


## Exportar o notebook

Para exportar em HTML ou PDF:

1. Baixe o arquivo `.ipynb`
2. No terminal ou no Colab/Jupyter, use:

```bash
jupyter nbconvert --to html nome_do_arquivo.ipynb
```

Depois, abra o HTML no navegador e imprima em PDF.


In [ ]:
!jupyter nbconvert --to html nome_do_arquivo.ipynb

[NbConvertApp] Converting notebook nome_do_arquivo.ipynb to html
[NbConvertApp] ERROR | Notebook JSON is invalid: Additional properties are not allowed ('metadata' was unexpected)

Failed validating 'additionalProperties' in stream:

On instance['cells'][41]['outputs'][0]:
{'metadata': {'tags': None},
 'name': 'stderr',
 'output_type': 'stream',
 'text': '100%|██████████| 170M/170M [00:05<00:00, 28.5MB/s]\n'}
[NbConvertApp] Writing 632440 bytes to nome_do_arquivo.html
